In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import requests

url = "https://huggingface.co/datasets/stanfordnlp/imdb/resolve/main/plain_text/train-00000-of-00001.parquet"

r = requests.get(url, stream=True, timeout=20)
print(r.status_code)

200


In [ ]:
!wget https://huggingface.co/datasets/stanfordnlp/imdb/resolve/main/plain_text/train-00000-of-00001.parquet

--2026-07-16 09:40:27--  https://huggingface.co/datasets/stanfordnlp/imdb/resolve/main/plain_text/train-00000-of-00001.parquet
Resolving huggingface.co (huggingface.co)... 18.164.174.17, 18.164.174.23, 18.164.174.55, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.17|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cas-bridge.xethub.hf.co/xet-bridge-us/621ffdd236468d709f181e77/7d30bb6e634c350424a4db373a4f40e0ce910f2c78809b7d6bb968ede8249396?Expires=1784198427&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzYyMWZmZGQyMzY0NjhkNzA5ZjE4MWU3Ny83ZDMwYmI2ZTYzNGMzNTA0MjRhNGRiMzczYTRmNDBlMGNlOTEwZjJjNzg4MDliN2Q2YmI5NjhlZGU4MjQ5Mzk2KiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4NDE5ODQyN319fV19&Signature=MEUCIQDvlzxDDdElqKnlQfvNe9YPxzZkucLmDdyt6zH7EsS4bgIgcaTg7aDpivTSdIw6xqVLl4Njke6f7RMq1m91UkPKY5A_&Key-Pair-Id=K3EPXBYC3CKDRZ&X-Xet-Cas-Uid=public&response-content-dispo

In [ ]:
!pip install -q transformers datasets evaluate accelerate

In [ ]:
import transformers
import datasets
import huggingface_hub

print(transformers.__version__)
print(datasets.__version__)
print(huggingface_hub.__version__)

Load the dataset

In [ ]:
from datasets import load_dataset

imdb = load_dataset('stanfordnlp/imdb')

In [ ]:
imdb['test'][0]

{'text': 'I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn\'t match the background, and painfully one-dimensional characters cannot be overcome with a \'sci-fi\' setting. (I\'m sure there are those of you out there who think Babylon 5 is good sci-fi TV. It\'s not. It\'s clichéd and uninspiring.) While US viewers might like emotion and character development, sci-fi is a genre that does not take itself seriously (cf. Star Trek). It may treat important issues, yet not as a serious philosophy. It\'s really difficult to care about the characters here as they are not simply foolish, just missing a spark of life. Their actions and reactions are wooden and predictable, often painful to watch. The makers of Earth KNOW it\'s rubbish as 

#Preprocess

Using DistilBert's AutoTokenizer to process the text




In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('distilbert/distilbert-base-uncased')

In [ ]:
tokenizer("I am Bipasna")

{'input_ids': [101, 1045, 2572, 12170, 19707, 2532, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

In [ ]:
tokens = tokenizer.tokenize("I am Bipasna")
print(tokens)

['i', 'am', 'bi', '##pas', '##na']


In [ ]:
def preprocess_function(examples):
  return tokenizer(examples['text'], truncation=True)

To apply the preprocess_function over the entire dataset, using Huggingface map function
"map()" function: take every example In the dataset, apply function to it AND RETURN new dataset.
Instead of writing loop ourselves we use "map()" which basically loops itself.

In [ ]:
tokenized_imdb = imdb.map(preprocess_function, batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Neural Networks can't process the input with variable length.
It requires same length of tokens.
For eg:
- Sentence 1: I love AI.
- Sentence 2: The cat is sat on the mat.
- Sentence 3: Bad
- Sentence 1
[101, 1045, 2293, 9932, 102]
Length = 5

- Sentence 2
[101, 2023, 3185, 2003, 7078, 6429, 102]
Length = 7

- Sentence 3
[101, 2919, 102]
Length = 3

Here comes the concept of Padding, padding is simply means adding special token called [PAD] until all sequences have the same length.

Suppose the longest sentece has length of 7 then all other sentence are padded and made to 7. i.e.
- Sentence 1: [101, 1045, 2293, 9932, 102, 0, 0] -> Length: 7
- Sentence 3: [101, 1045, 2293, 0, 0, 0, 0] -> Length: 7

Since the entire dataset is being padded at once then it creates the adding of large number of useless token based on the longest length size.
For eg: the token of 40 size and longest sequence if of 512 token size
then it will add large number of useless token, waste of gpu, computation.
So, Dynamic Padding is introduced where the padding is done batch wise.
The batch one might have the difference of 20 tokens then it will add 20 tokens to all other tokens.
The batch 2 might have the difference of 212 tokens then it will do so on.
Based on the batches even in the same dataset.
So the tensors or the NN will be able to process batch wise without problem.

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Evaluate during training

In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")

In [ ]:
import numpy as np

def compute_metrics(eval_pred):
  predictions, labels = eval_pred
  predictions = np.argmax(predictions, axis=1)
  return accuracy.compute(predictions=predictions, references=labels)

Train

In [ ]:
id2label = {0:"NEGATIVE", 1:"POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased", num_labels=2, id2label=id2label, label2id=label2id)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
training_args = TrainingArguments(
    output_dir="text_classification_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_imdb["train"],
    eval_dataset=tokenized_imdb["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.224754,0.194035,0.925960


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.224754,0.194035,0.925960
2,0.149676,0.228177,0.930920


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3126, training_loss=0.2082105723238876, metrics={'train_runtime': 3123.5673, 'train_samples_per_second': 16.007, 'train_steps_per_second': 1.001, 'total_flos': 6556904415524352.0, 'train_loss': 0.2082105723238876, 'epoch': 2.0})

In [21]:
trainer.save_model("model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [23]:
tokenizer.save_pretrained("model")

('model/tokenizer_config.json', 'model/tokenizer.json')

In [17]:
text = "This was a masterpiece. Not completely faithful to the books, but enthralling from beginning to end. Might be my favorite of the three."

In [27]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model="model", tokenizer="model" )

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [28]:
classifier("This was a masterpiece. Not completely faithful to the books, but enthralling from beginning to end. Might be my favorite of the three.")

[{'label': 'POSITIVE', 'score': 0.9918495416641235}]

In [29]:
reviews = [
    "I loved this movie!",
    "Worst film ever.",
    "It was okay, nothing special."
]

classifier(reviews)

[{'label': 'POSITIVE', 'score': 0.9925732612609863},
 {'label': 'NEGATIVE', 'score': 0.9781112670898438},
 {'label': 'NEGATIVE', 'score': 0.9072190523147583}]

the pipeline automatically did all of these steps:

1. Load the tokenizer used in the specified model.
2. Tokenize the text.
3. Convert it to PyTorch tensors.
4. Run the model.
5. Compute the logits.
6. Find the class with the highest score (argmax).
7. Convert the class ID into a readable label.
8. Return the prediction (and confidence score).

So, the above behind the scenes is shown below :

In [31]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("model")
inputs = tokenizer(text, return_tensors="pt")

In [33]:
from transformers import AutoModelForSequenceClassification
import torch

model = AutoModelForSequenceClassification.from_pretrained("model")
with torch.no_grad():
    logits = model(**inputs).logits

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [34]:
predicted_class_id = logits.argmax().item()
model.config.id2label[predicted_class_id]

'POSITIVE'